# 使用双端口网络分析仪测量多端口器件

## 简介

在微波测量中，通常需要使用一个 m 端口网络分析仪 ($m<n$) 来测量一个 n 端口器件。<img src="nports_with_2ports.svg"/>可以通过将每个未测量的端口连接到一个匹配的负载，并假设反射功率可以忽略不计，来实现这一点。通过多次测量，就可以重建原始的 n 端口器件。本示例的第一部分展示了这种方法。但是，在某些情况下，这种方法可能无法提供最准确的结果，或者在所有测量环境中都不可行。或者，有时无法为所有端口提供匹配的负载。本示例的第二部分提供了一种优雅的解决方案，使用阻抗重整化。我们将称其为 *Tippet 技术*，因为这个名字听起来不错。

In [ ]:
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np

import skrf as rf
from skrf.network import connect, subnetwork

%matplotlib inline

rf.stylely()

## 匹配端口假设你有一个 2 端口 VNA。为了测量一个 n 端口网络，你需要至少进行 $p=n(n-1)/2$ 次测量，即不同端口对之间的测量（n 个端口的总唯一配对数）。例如，假设我们要用一个 2 端口 VNA 测量一个 3 端口网络。至少需要进行 3 次测量：端口 1 和端口 2 之间、端口 2 和端口 3 之间以及端口 1 和端口 3 之间。我们将假设这些测量结果随后被转换为三个 2 端口 `Network`。为了构建完整的 3 端口 `Network`，需要向 scikit-rf 内置函数 `n_twoports_2_nport` 提供一个包含这 3 个（子）网络的列表。虽然列表中测量的顺序并不重要，但请注意定义这些子网络的 `Network.name` 属性，使其包含端口索引，例如，对于端口 1 和端口 2 之间的测量，可以是 `p12`；对于端口 2 和端口 3 之间的测量，可以是 `p23`，依此类推。假设我们要测量一个 T 型网络：

In [ ]:
tee = rf.data.tee
print(tee)

为了演示，我们将通过从原始网络中提取 3 个子集，即 3 个子网络，来“模拟”这 3 个不同的测量值：

In [ ]:
# 2 port Networks as if one measures the tee with a 2 ports VNA
tee12 = subnetwork(tee, [0, 1])  # 2 port Network btw ports 1 & 2, port 3 being matched
tee23 = subnetwork(tee, [1, 2])  # 2 port Network btw ports 2 & 3, port 1 being matched
tee13 = subnetwork(tee, [0, 2])  # 2 port Network btw ports 1 & 3, port 2 being matched

实际上，这三个 skrf.Network 对象来自三次不同的测量，每次测量使用不同的端口对，未使用的端口应进行适当的匹配。在使用 `n_twoports_2_nport` 函数之前，必须通过设置 `Network.name` 属性来定义这些子集的名称，以便该函数能够知道哪个子集对应于哪个测量。

In [ ]:
tee12.name = 'tee12'
tee23.name = 'tee23'
tee13.name = 'tee13'

现在，我们可以使用这三个二端口子网络构建一个三端口网络：

In [ ]:
ntw_list = [tee12, tee23, tee13]
tee_rebuilt = rf.network.n_twoports_2_nport(ntw_list, nports=3)
print(tee_rebuilt)

In [ ]:
# this is an ideal example, both Network are thus identical
print(tee == tee_rebuilt)

## 蒂佩特技术本示例演示了对以下论文中描述的技术进行数值测试：*“使用 2 端口网络分析仪测量多端口器件散射矩阵的严格技术”*[1]。在*蒂佩特技术*中，多个子网络以类似于之前的方式进行测量，但端口端接不再假定为匹配状态。相反，端接只需要已知，并且最多只能有一个完全反射。因此，通常 $|\Gamma| \ne 1$。在测量过程中，每个端口都使用一致的端接电阻。因此，当未进行测量时，端口 1 始终端接于 $Z_1$。一旦测量完成，每个子网络都会重新归一化到这些端口阻抗。请思考一下。最后，构建复合网络，然后可以将其重新归一化到所需的系统阻抗，例如 50 欧姆。* [1] J. C. Tippet 和 R. A. Speciale，《使用 2 端口网络分析仪测量多端口器件散射矩阵的严格技术》，《IEEE 微波理论与技术汇刊》，第 30 卷，第 5 期，第 661-666 页，1982 年 5 月。

## Tippet 技术的概要

以下面的示例 [1] 为例，使用 2 端口网络分析仪测量 4 端口网络。技术概要：1. 校准 2 端口网络分析仪。2. 获取四个已知的终端阻抗 ($Z_1, Z_2, Z_3, Z_4$)。最多只能有一个终端阻抗的反射系数 $|\Gamma| = 1$。3. 测量所有 2 端口子网络的组合（共有 6 个）。每个未当前测量的端口都必须与其对应的负载端接。4. 将每个子网络重新归一化到用于在其未测量时进行端接的负载的阻抗。5. 构建复合的 4 端口网络，并重新归一化到 VNA 的阻抗。

## 实现方法

首先，我们创建一个 Media 对象，该对象用于生成用于测试的网络。我们将使用 WR-10 矩形波导。

In [ ]:
wg = rf.instances.wr10

接下来，我们将生成一个随机的 4 端口网络，作为被测设备 (DUT)，我们尝试使用我们的 2 端口网络分析仪对其进行测量。

In [ ]:
dut = wg.random(n_ports  = 4,name= 'dut')
dut

现在，我们需要定义在未进行测量时用于终止每个端口的负载，如[1]中所述，负载的反射系数 $|\Gamma|$ 不应超过 1，即不能有超过一个负载具有完全反射。

In [ ]:
loads = [wg.load(.1+.1j),
         wg.load(.2-.2j),
         wg.load(.3+.3j),
         wg.load(.5),
         ]
# construct the impedance array, of shape FXN
z_loads = np.array([k.z.flatten() for k in loads]).T


创建所需的测量端口组合。使用 2 端口 VNA 测量 4 端口器件需要进行 6 种不同的测量。一般来说，对于在 2 端口 VNA 上的 n 端口 DUT，测量的次数为 $n\choose 2$。

In [ ]:
ports = np.arange(dut.nports)
port_combos = list(combinations(ports, 2))
port_combos

现在开始操作。好的，我们遍历端口组合，并将负载连接到正确的位置，模拟实际测量。每个原始子网络测量值都会被保存，同时也会保存重新归一化的子网络。最后，我们将结果存储到 4 端口组合网络中。

In [ ]:
composite = wg.match(nports = 4)  # composite network, to be filled.
measured,measured_renorm = {},{}  # measured subnetworks and renormalized sub-networks

# ports  `a` and `b` are the ports we will connect the VNA too
for a,b in port_combos:
    # port `c` and `d` are the ports which we will connect the loads too
    c,d =ports[(ports!=a)& (ports!=b)]

    # determine where `d` will be on four_port, after its reduced to a three_port
    e = np.where(ports[ports!=c]==d)[0][0]

    # connect loads
    three_port = connect(dut,c, loads[c],0)
    two_port =  connect(three_port,e, loads[d],0)

    # save raw and renormalized 2-port subnetworks
    measured['%i%i'%(a,b)] = two_port.copy()
    two_port.renormalize(np.c_[z_loads[:,a],z_loads[:,b]])
    measured_renorm['%i%i'%(a,b)] = two_port.copy()

    # stuff this 2-port into the composite 4-port
    for i,m in enumerate([a,b]):
        for j,n in enumerate([a,b]):
            composite.s[:,m,n] = two_port.s[:,i,j]

    # properly copy the port impedances
    composite.z0[:,a] = two_port.z0[:,0]
    composite.z0[:,b] = two_port.z0[:,1]

# finally renormalize from load z0 to 50 ohms
composite.renormalize(50)

## 结果

### 自洽性

请注意，对 2 端口子网络的 6 次测量会产生 24 个 s 参数，而我们只需要 16 个。这是因为每个反射 s 参数都会被测量三次。如 [1] 中所示，我们将利用这些冗余测量结果来验证我们的准确性。重新归一化的网络存储在一个字典中，字典的键是基于端口索引的。从这里可以看出，每个网络都已重新归一化到相应的 z0。

In [ ]:
measured_renorm

绘制所有三个原始的 $S_{11}$ 测量值，我们可以看到它们之间存在差异。这些图与参考文献 [1] 中的图 5 和图 7 相对应。

In [ ]:
s11_set = rf.networkSet.NetworkSet([measured[k] for k in measured if k[0]=='0'])

plt.figure(figsize = (8,4))
plt.subplot(121)
s11_set .plot_s_db(0,0)
plt.subplot(122)
s11_set .plot_s_deg(0,0)
plt.tight_layout()

然而，重新归一化的测量结果完全一致。这些图与参考文献 [1] 中的图 6 和图 8 对应。

In [ ]:
s11_set = rf.networkSet.NetworkSet([measured_renorm[k] for k in measured_renorm if k[0]=='0'])

plt.figure(figsize = (8,4))
plt.subplot(121)
s11_set .plot_s_db(0,0)
plt.subplot(122)
s11_set .plot_s_deg(0,0)
plt.tight_layout()

### 精度测试

确保我们构建的组合网络与待测器件（DUT）相同。

In [ ]:
composite == dut

很好！接近多少？

In [ ]:
sum((composite - dut).s_mag)

太棒了！

## 实际应用

这有很多种应用方式。在波导中，可以先进行标准的双端口校准（如 TRL），然后测量辐射开路后的散射参数。接着，使用 *Tippets 技术*，在不进行测量的同时，让每个端口都保持开路状态。这样就无需购买大量的负载。这不是很棒吗？

## 更复杂的仿真

In [ ]:
def tippits(dut, gamma, noise=None):
    """simulate tippits technique on a 4-port dut.
    """
    ports = np.arange(dut.nports)
    port_combos = list(combinations(ports, 2))

    loads = [wg.load(gamma) for k in ports]

    # construct the impedance array, of shape FXN
    z_loads = np.array([k.z.flatten() for k in loads]).T
    composite = wg.match(nports = dut.nports)  # composite network, to be filled.

    # ports  `a` and `b` are the ports we will connect the VNA too
    for a,b in port_combos:
        # port `c` and `d` are the ports which we will connect the loads too
        c,d =ports[(ports!=a)& (ports!=b)]

        # determine where `d` will be on four_port, after its reduced to a three_port
        e = np.where(ports[ports!=c]==d)[0][0]

        # connect loads
        three_port = connect(dut,c, loads[c],0)
        two_port =  connect(three_port,e, loads[d],0)

        if noise is not None:
            two_port.add_noise_polar(*noise)
        # save raw and renormalized 2-port subnetworks
        measured['%i%i'%(a,b)] = two_port.copy()
        two_port.renormalize(np.c_[z_loads[:,a],z_loads[:,b]])
        measured_renorm['%i%i'%(a,b)] = two_port.copy()

        # stuff this 2-port into the composite 4-port
        for i,m in enumerate([a,b]):
            for j,n in enumerate([a,b]):
                composite.s[:,m,n] = two_port.s[:,i,j]

        # properly copy the port impedances
        composite.z0[:,a] = two_port.z0[:,0]
        composite.z0[:,b] = two_port.z0[:,1]

    # finally renormalize from load z0 to 50 ohms
    composite.renormalize(50)

    return composite

In [ ]:
dut = wg.random(4)

def er(gamma, *args):
    return max(abs(tippits(dut, rf.mathFunctions.db_2_mag(gamma),*args).s_db-dut.s_db).flatten())

gammas = np.linspace(-40,-0.1,11)

plt.title(r'Error vs $|\Gamma|$')
plt.plot(gammas, [er(k) for k in gammas])
plt.semilogy()
plt.xlabel(r'$|\Gamma|$ of Loads (dB)')
plt.ylabel('Max Error in DUT\'s dB(S)')

plt.figure()
noise = (1e-5,.1)
plt.title(r'Error vs $|\Gamma|$ with reasonable noise')
plt.plot(gammas, [er(k, noise) for k in gammas])
plt.semilogy()
plt.xlabel(r'$|\Gamma|$ of Loads (dB)')

plt.ylabel('Max Error in DUT\'s dB(S)')